In [1]:
# ensure classes imported from .py files are dynamically updated
%load_ext autoreload
%autoreload 2

# plot matplots nicely
%matplotlib inline  

In [2]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean
from cartopy.feature import LAND
from stericheight.data_handler import *
from glob import glob
from OceanDataStore import OceanDataCatalog

In [3]:
catalog = OceanDataCatalog(catalog_name="noc-model-stac")

In [5]:
catalog.available_collections

['noc-rapid-evolution', 'noc-npd-jra55', 'noc-npd-era5']

In [6]:
dsu = catalog.open_dataset(id='noc-npd-era5/npd-eorca12-era5v1/gn/U1m_4d',variable_names=['uo'])
dsv = catalog.open_dataset(id='noc-npd-era5/npd-eorca12-era5v1/gn/V1m_4d',variable_names=['vo'])

ValueError: Unsupported media type application/vnd.zarr+icechunk for Item asset.

In [ ]:
sla_ = SLA(deg=0.25).da

minla = -68
maxla = -63

extent_ac = [138,152,minla,maxla]
extent_ea=[60,160,-70,-60]


In [ ]:
def prep_nemo(ds_in,t_lims,lat_lims,lon_lims):
    # crop to time & make monthly
    ds_renamed = ds_in.rename({'time_counter':'time'})
    ds_cropped = ds_renamed.sel(time=slice(t_lims[0],t_lims[1])).resample({'time':'MS'}).mean()

    # crop to region
    lats = ds_cropped.nav_lat.load()
    lons = ds_cropped.nav_lon.load()

    mask = (lats >= lat_lims[0]) & (lats <= lat_lims[1]) & (lons >= lon_lims[0]) & (lons <= lon_lims[1])

    return ds_cropped.where(mask,drop=True)

In [ ]:
nu = prep_nemo(dsu,[sla_.time[0],sla_.time[-1]],[minla,maxla],[extent_ac[0],extent_ac[1]])
nv = prep_nemo(dsv,[sla_.time[0],sla_.time[-1]],[minla,maxla],[extent_ac[0],extent_ac[1]])

In [ ]:
nu.uo.where(navnu.integrate('depthu')

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 227, depthu: 75, y: 146, x: 168)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2003-01-01 2003-02-01 ... 2021-11-01
  * depthu   (depthu) float32 300B 0.5058 1.556 2.668 ... 5.698e+03 5.902e+03
  * y        (y) int64 1kB 915 916 917 918 919 920 ... 1056 1057 1058 1059 1060
  * x        (x) int64 1kB 780 781 782 783 784 785 ... 942 943 944 945 946 947
    nav_lon  (y, x) float64 196kB 138.0 138.1 138.2 138.3 ... 151.8 151.9 152.0
    nav_lat  (y, x) float64 196kB -67.99 -67.99 -67.99 ... -63.0 -63.0 -63.0
Data variables:
    uo       (time, depthu, y, x) float32 2GB dask.array<chunksize=(1, 25, 146, 168), meta=np.ndarray>
Attributes:
    name:         OUTPUT/eORCA12_1m_grid_U
    description:  ocean U grid variables
    title:        ocean U grid variables
    Conventions:  CF-1.6
    timeStamp:    2025-Jul-22 16:18:45 GMT
    uuid:         0ec281fd-971f-4146-83c6-ce785ca08bdf